# Практика №2.

## Задание

1. Провести статистическую обработку текста.
2. Определить энтропию, приходящуюся в среднем на одну букву, длину кода при равномерном кодировании и избыточность.
3. Построить схему алфавитного кодирования для однобуквенных сочетаний методом Шеннона-Фано. Найти среднюю длину элементарного кода, эффективность сжатия. Закодировать текст. Декодировать текст.
4. Построить схему алфавитного кодирования для двухбуквенных сочетаний методом Шеннона-Фано. Найти среднюю длину элементарного кода, эффективность сжатия, сравнить с результатами для однобуквенных сочетаний. Закодировать текст. Декодировать
текст.

In [57]:
import math
import pickle
import string
from collections import Counter

## Обработка входного текста

In [58]:
with open('some_text.txt', 'r', encoding='utf-8') as file:
    text = file.read()

In [59]:
text = text.lower()  # Приведение к нижнему регистру
text = text.translate(str.maketrans('', '', string.punctuation + '\n'))  # Удаление знаков препинания
text = text.replace(" ", "")  # Удаление пробелов

## Основные функции

In [60]:
# Подсчет частот символов
def calculate_frequencies(text: str) -> Counter[str]:
    single_char_freq = Counter(text)  # Частоты однобуквенных сочетаний
    bigrams = [text[i:i+2] for i in range(len(text)-1)]  # Двухбуквенные сочетания
    bigram_freq = Counter(bigrams)
    return single_char_freq, bigram_freq

In [61]:
# Вычисление энтропии текста
def calculate_entropy(freq_dict: dict, total_count: int) -> float:
    entropy = 0
    for freq in freq_dict.values():
        p = freq / total_count
        entropy -= p * math.log2(p)
    return entropy

In [62]:
# Алгоритм Шеннона-Фано
class ShannonFanoNode:
    def __init__(self, probability: float, data: str, left=None, right=None) -> None:
        self.probability = probability
        self.data = data
        self.left = left
        self.right = right

def convert_to_shanon_nodes(freq_dict: dict, total_count: int) -> list[ShannonFanoNode]:
    nodes = []
    for char, count in freq_dict.items():
        probability = count / total_count
        node = ShannonFanoNode(probability, char)
        nodes.append(node)
    nodes.sort(key=lambda node: node.probability, reverse=True)
    return nodes


def build_shannon_fano_tree(node_list: list[ShannonFanoNode]) -> ShannonFanoNode:
    if len(node_list) == 1:
        return node_list[0]

    total_prob = sum(node.probability for node in node_list)
    split_index = 0
    cumulative_prob = 0
    
    for i, node in enumerate(node_list):
        if cumulative_prob + node.probability > total_prob / 2:
            break
        cumulative_prob += node.probability
        split_index = i
    
    left = build_shannon_fano_tree(node_list[:split_index + 1])
    right = build_shannon_fano_tree(node_list[split_index + 1:])

    return ShannonFanoNode("", total_prob, left, right)


def generate_shannon_codes(node: ShannonFanoNode, prefix="", codebook={}):
    if node.left is None and node.right is None:
        codebook[node.data] = prefix
    else:
        generate_shannon_codes(node.left, prefix + "0", codebook)
        generate_shannon_codes(node.right, prefix + "1", codebook)

In [63]:
# Кодирование текста
def encode(message: str, codebook: dict[str, str]) -> str:
    encoded_message = ""
    for letter in message:
        encoded_message += codebook[letter]

    return encoded_message

# Декодирование текста
def decode(encoded_message: str, codebook: dict[str, str]) -> str:
    decoding_codebook = dict((code, char) for char, code in codebook.items())
    decoded_message = ""
    bits = ""
    for bit in encoded_message:
        bits += bit
        if decoding_codebook.get(bits) is not None:
            decoded_message += decoding_codebook[bits]
            bits = ""

    return decoded_message

In [64]:
# Кодирование текста с сохранением в бинарный файл
def encode_to_file(message: str, codebook: dict[str, str], filename: str) -> None:
    # Создание бинарного представления закодированного сообщения
    encoded_bits = ''.join(codebook[letter] for letter in message)
    
    # Преобразование битов в байты
    byte_array = bytearray()
    for i in range(0, len(encoded_bits), 8):
        byte = int(encoded_bits[i:i + 8], 2)  # Преобразуем 8 бит в целое число
        byte_array.append(byte)
    
    # Сохранение закодированного сообщения и кодовой книги в бинарный файл
    with open(filename, 'wb') as f:
        pickle.dump((byte_array, codebook), f)

# Декодирование текста из бинарного файла
def decode_from_file(filename: str) -> str:
    # Загрузка закодированного сообщения и кодовой книги из бинарного файла
    with open(filename, 'rb') as f:
        byte_array, codebook = pickle.load(f)

    # Преобразование байтов в битовую строку
    encoded_bits = ''.join(f'{byte:08b}' for byte in byte_array)
    
    decoding_codebook = {code: char for char, code in codebook.items()}
    decoded_message = ""
    bits = ""
    for bit in encoded_bits:
        bits += bit
        if bits in decoding_codebook:
            decoded_message += decoding_codebook[bits]
            bits = ""

    return decoded_message

### 1. Провести статистическую обработку текста.

In [65]:
single_char_freq, bigram_freq = calculate_frequencies(text)

In [66]:
# Генерация вероятностной схемы
print(" char | n     | w     ")
print("------+-------+-------")
for char, count in single_char_freq.items():
    print(f"{char:>5} | {count:>5} | {(count / len(text)):>6.4f}")

 char | n     | w     
------+-------+-------
    h |     1 | 0.1000
    e |     1 | 0.1000
    l |     3 | 0.3000
    o |     2 | 0.2000
    w |     1 | 0.1000
    r |     1 | 0.1000
    d |     1 | 0.1000


### 2. Определить энтропию, приходящуюся в среднем на одну букву, длину кода при равномерном кодировании и избыточность.

In [67]:
# Энтропия
single_char_entropy = calculate_entropy(single_char_freq, len(text))
print(f"Энтропия: {single_char_entropy:.4f} бит на символ")

# Длина кода при равномерном кодировании
uniform_code_length = math.log2(len(single_char_freq))
uniform_code_length_ceil = math.ceil(uniform_code_length)
print(f"Длина кода при равномерном кодировании: {uniform_code_length_ceil} бит")

# Избыточность
redundancy = uniform_code_length - single_char_entropy
print(f"Избыточность: {redundancy:.4f}")

redundancy_2 = 1 - uniform_code_length / uniform_code_length_ceil
print(f"Избыточность округления: {redundancy_2:.4f}")

redundancy_3 = 1 - single_char_entropy / math.log2(len(single_char_freq))
print(f"Информационная избыточность: {redundancy_3:.4f}")

Энтропия: 2.6464 бит на символ
Длина кода при равномерном кодировании: 3 бит
Избыточность: 0.1609
Избыточность округления: 0.0642
Информационная избыточность: 0.0573


### 3. Построить схему алфавитного кодирования для однобуквенных сочетаний методом Шеннона-Фано. Найти среднюю длину элементарного кода, эффективность сжатия. Закодировать текст. Декодировать текст.

In [68]:
# Генерация схемы кодирования по алгоритму Шеннона-Фано
shanon_nodes = convert_to_shanon_nodes(single_char_freq, len(text))
shanon_tree = build_shannon_fano_tree(shanon_nodes)
codebook = {}
generate_shannon_codes(shanon_tree, codebook=codebook)

In [69]:
# Схема кодирования по алгоритму Шеннона-Фано
print(" char | code")
print("------+-----------")
for char, code in codebook.items():
    print(f"{char:>5} | {code}")

 char | code
------+-----------
    l | 0
    o | 100
    h | 101
    e | 1100
    w | 1101
    r | 1110
    d | 1111


In [70]:
# Средняя длина кодового слова
avg_code_length = sum([single_char_freq[char] / len(text) * len(codebook[char]) for char in single_char_freq])
print(f"Средняя длина кодового слова: {avg_code_length:.4f}")

# Коэффициент эффективности
efficiency_factor = single_char_entropy / avg_code_length
print(f"Коэффициент эффективности: {efficiency_factor:.4f}")

Средняя длина кодового слова: 2.8000
Коэффициент эффективности: 0.9452


In [71]:
# Кодирование текста
encoded_text = encode(text, codebook)
print("Закодированный текст:", encoded_text)

# Декодирование текста
decoded_text = decode(encoded_text, codebook)
print("Декодированный текст:", decoded_text)

Закодированный текст: 1011100001001101100111001111
Декодированный текст: helloworld


In [72]:
# Кодирование текста в бинарный файл
encode_to_file(text, codebook, "archived.bin")

# Декодирование текста из бинарного файла
decoded_text = decode_from_file("archived.bin")
print("Декодированный текст:", decoded_text)

Декодированный текст: helloworllllld


In [73]:
compression_factor = len(encoded_text) / (len(text) * 8) * 100
print(f"Коэффициент сжатия: {compression_factor:.4f}%")

Коэффициент сжатия: 35.0000%


### Построить схему алфавитного кодирования для двухбуквенных сочетаний методом Шеннона-Фано. Найти среднюю длину элементарного кода, эффективность сжатия, сравнить с результатами для однобуквенных сочетаний. Закодировать текст. Декодировать текст.

In [74]:
# Генерация схемы кодирования по алгоритму Шеннона-Фано
bigram_shanon_nodes = convert_to_shanon_nodes(bigram_freq, len(text)-1)
bigram_shanon_tree = build_shannon_fano_tree(bigram_shanon_nodes)
bigram_codebook = {}
generate_shannon_codes(bigram_shanon_tree, codebook=bigram_codebook)

In [75]:
# Схема кодирования по алгоритму Шеннона-Фано
print(" chars | code")
print("-------+-----------")
for chars, code in bigram_codebook.items():
    print(f"{chars:>6} | {code}")

 chars | code
-------+-----------
    he | 000
    el | 001
    ll | 010
    lo | 011
    ow | 100
    wo | 101
    or | 110
    rl | 1110
    ld | 1111


In [76]:
# Средняя длина кодового слова
avg_code_length = sum([bigram_freq[char] / (len(text)-1) * len(bigram_codebook[char]) for char in bigram_freq])
print(f"Средняя длина кодового слова: {avg_code_length:.4f}")

# Коэффициент эффективности
bigram_entropy = calculate_entropy(bigram_freq, len(text)-1)
bigram_efficiency_factor = bigram_entropy / avg_code_length
print(f"Коэффициент эффективности: {bigram_efficiency_factor:.4f}")

Средняя длина кодового слова: 3.2222
Коэффициент эффективности: 0.9838


In [77]:
# Кодирование текста
bigram_encoded_text = encode(text, codebook)
print("Закодированный текст:", bigram_encoded_text)

# Декодирование текста
bigram_decoded_text = decode(bigram_encoded_text, codebook)
print("Декодированный текст:", bigram_decoded_text)

Закодированный текст: 1011100001001101100111001111
Декодированный текст: helloworld


In [78]:
# Кодирование текста в бинарный файл
encode_to_file(text, codebook, "archived.bin")

# Декодирование текста из бинарного файла
bigram_decoded_text = decode_from_file("archived.bin")
print("Декодированный текст:", bigram_decoded_text)

Декодированный текст: helloworllllld


In [79]:
bigram_compression_factor = len(bigram_encoded_text) / (len(text) * 8) * 100
print(f"Коэффициент сжатия: {bigram_compression_factor:.4f}%")

Коэффициент сжатия: 35.0000%
